# A: Preprocessing

## 1. Imports and paths

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.ticker import MultipleLocator, FuncFormatter, FormatStrFormatter, AutoLocator, PercentFormatter
import matplotlib.transforms as mtransforms
from matplotlib.lines import Line2D
from matplotlib.font_manager import FontProperties
from matplotlib.patches import Rectangle
from matplotlib.legend_handler import HandlerTuple

from Bio import SeqIO
try:
    import ahocorasick
except ImportError:
    ahocorasick = None

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists() and (PROJECT_DIR.parent / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
RECIPE_DIR = DATA_DIR / "recipes"
PEPTIDE_TABLE_DIR = DATA_DIR / "peptide_tables"
TAXON_TABLE_DIR = DATA_DIR / "taxon_tables"
FASTA_DIR = DATA_DIR / "fasta"
ALPHAPEPT_HDF_DIR = DATA_DIR / "alphapept_hdf"


## 2. Constants and file maps

In [ ]:
KLEINER_CONDITIONS = ["P", "U", "C"]
KLEINER_METHODS = ["Uniques", "Uniform", "Clades", "TaxonLFQ"]

KLEINER_COMPOSITION_FILES = {
    "P": "Composition_Of_EQUAL_PROTEIN_AMOUNT_Community.tab",
    "U": "Composition_Of_UNEVEN_Community.tab",
    "C": "Composition_Of_EQUAL_CELL_NUMBER_Community.tab",
}

KLEINER_METHOD_FILES = {
    "Uniques": {
        "P": "Kleiner_uniques_P.parquet",
        "U": "Kleiner_uniques_U.parquet",
        "C": "Kleiner_uniques_C.parquet",
    },
    "Uniform": {
        "P": "Kleiner_uniform_P.parquet",
        "U": "Kleiner_uniform_U.parquet",
        "C": "Kleiner_uniform_C.parquet",
    },
    "Clades": {
        "P": "Kleiner_Clades_P.parquet",
        "U": "Kleiner_Clades_U.parquet",
        "C": "Kleiner_Clades_C.parquet",
    },
    "TaxonLFQ": {
        "P": "Kleiner_TaxonLFQ_direct_P.parquet",
        "U": "Kleiner_TaxonLFQ_direct_U.parquet",
        "C": "Kleiner_TaxonLFQ_direct_C.parquet",
    },
}

KLEINER_LABELS = {
    "P": [
        "PD", "BS", "PaD", "AK199", "KF7", "CV", "ATN", "SMS",
        "Cup", "Pfl", "BXL", "137", "259", "Am2", "LT2a",
        "K12", "LT2b", "LT2c", "HB2", "CRH", "M13", "F2", "P22",
        "F0", "ES18", "841", "VF", "NV",
    ],
    "U": [
        "PD", "BS", "PaD", "AK199", "KF7", "CV", "ATN", "SMS",
        "Cup", "Pfl", "BXL", "DVH", "137", "259", "Am2", "LT2a",
        "K12", "LT2b", "LT2c", "HB2", "CRH", "M13", "F2", "P22",
        "F0", "ES18", "841", "VF", "Ne1", "Nu1", "Nm1", "NV",
    ],
    "C": [
        "PD", "BS", "PaD", "AK199", "KF7", "CV", "ATN", "SMS",
        "Cup", "Pfl", "BXL", "137", "259", "Am2", "LT2a",
        "K12", "LT2b", "LT2c", "HB2", "CRH", "M13", "F2", "P22",
        "F0", "ES18", "841", "VF", "NV",
    ],
}

KLEINER_ORDER_COMPLETE = {
    "P": [
        "NV", "841", "VF", "ATN", "AK199", "PaD", "Am2", "K12", "LT2",
        "KF7", "PD", "Pfl", "SMS", "BXL", "Cup", "CV", "CRH",
        "259", "137", "BS", "HB2", "ES18", "cRAP", "F0", "P22", "F2", "M13",
    ],
    "U": [
        "NV", "841", "VF", "ATN", "AK199", "PaD", "Am2", "K12", "LT2",
        "KF7", "PD", "Pfl", "SMS", "BXL", "Cup", "Ne1", "Nm1", "Nu1", "DVH", "CV", "CRH",
        "259", "137", "BS", "HB2", "ES18", "cRAP", "F0", "F2", "P22", "M13",
    ],
    "C": [
        "NV", "841", "VF", "ATN", "AK199", "PaD", "Am2", "K12", "LT2",
        "KF7", "PD", "Pfl", "SMS", "BXL", "Cup", "CV", "CRH",
        "259", "137", "BS", "HB2", "ES18", "cRAP", "F0", "P22", "F2", "M13",
    ],
}

KLEINER_ORDER = [
    "NV", "841", "VF", "ATN", "AK199", "PaD", "Am2", "K12", "LT2",
    "KF7", "PD", "Pfl", "SMS", "BXL", "Cup", "CV", "CRH",
    "259", "137", "BS", "HB2",
]

KLEINER_CONTRASTS = {
    "P_vs_U": {"numerator": "P", "denominator": "U", "title": "P vs. U", "obs_label": r"P$_{\mathrm{obs}}$"},
    "P_vs_C": {"numerator": "P", "denominator": "C", "title": "P vs. C", "obs_label": r"P$_{\mathrm{obs}}$"},
    "C_vs_U": {"numerator": "C", "denominator": "U", "title": "C vs. U", "obs_label": r"C$_{\mathrm{obs}}$"},
}

ZHAO_SAMPLES = ["S1", "S2", "S3"]
ZHAO_METHODS = ["Uniques", "Uniform", "Clades", "TaxonLFQ"]
ZHAO_ORDER = ["Bfr", "Cbu", "Cfr", "Eas", "Eca", "Eco", "Efa", "Kae", "Kpn", "Lac", "Mmo", "Pae"]

ZHAO_METHOD_FILES = {
    "TaxonLFQ": {
        "S1": "Zhao_TaxonLFQ_noLFQ_S1.parquet",
        "S2": "Zhao_TaxonLFQ_noLFQ_S2.parquet",
        "S3": "Zhao_TaxonLFQ_noLFQ_S3.parquet",
    },
    "Uniques": {
        "S1": "Zhao_LFQ_uniques_S1.parquet",
        "S2": "Zhao_LFQ_uniques_S2.parquet",
        "S3": "Zhao_LFQ_uniques_S3.parquet",
    },
    "Uniform": {
        "S1": "Zhao_uniform_S1.parquet",
        "S2": "Zhao_uniform_S2.parquet",
        "S3": "Zhao_uniform_S3.parquet",
    },
    "Clades": {
        "S1": "Zhao_LFQ_clades_S1.parquet",
        "S2": "Zhao_LFQ_clades_S2.parquet",
        "S3": "Zhao_LFQ_clades_S3.parquet",
    },
}

ZHAO_CONTRASTS = {
    "S3/S1": {"numerator": "S3", "denominator": "S1", "title": "Sample 3 vs. Sample 1"},
    "S3/S2": {"numerator": "S3", "denominator": "S2", "title": "Sample 3 vs. Sample 2"},
    "S2/S1": {"numerator": "S2", "denominator": "S1", "title": "Sample 2 vs. Sample 1"},
}

ZHAO_EXPECTED_FC = {
    "S2/S1": {
        "Kpn": 1/5, "Bfr": 5/1, "Pae": 2/1, "Mmo": 1/5,
        "Eca": 5/2, "Cbu": 2/1, "Efa": 2/1, "Eas": 1/5,
        "Cfr": 5/2, "Lac": 2/5, "Eco": 3/2, "Kae": 2/1,
    },
    "S3/S1": {
        "Kpn": 2/5, "Bfr": 2/1, "Pae": 5/1, "Mmo": 2/5,
        "Eca": 1/2, "Cbu": 5/1, "Efa": 5/1, "Eas": 2/5,
        "Cfr": 1/2, "Lac": 1/5, "Eco": 1/2, "Kae": 5/1,
    },
}
ZHAO_EXPECTED_FC["S3/S2"] = {
    k: ZHAO_EXPECTED_FC["S3/S1"][k] / ZHAO_EXPECTED_FC["S2/S1"][k]
    for k in set(ZHAO_EXPECTED_FC["S2/S1"]) & set(ZHAO_EXPECTED_FC["S3/S1"])
}

ZHAO_AUTHOR_OFFSETS = {
    "S3/S1": [0.62, 0.81, 0.04, 0.03, 0.08, 0.06, 1.61, -1.62, 0.07, 0.01, 0.03, 1.98],
    "S3/S2": [0.01, 0.79, -0.02, 0.15, -0.02, 0.08, 0.25, -0.14, -1.31, 0.05, 0.39, 0.56],
    "S2/S1": [1.37, -0.23, 0.54, 0.01, 0.75, -0.17, 0.41, 0.0, 0.49, -0.04, -0.02, 0.31],
}

KLEINER_ALPHAPEPT_HDF_FILES = {
    "P": "results_Kleiner_P_LT2.hdf",
    "U": "results_Kleiner_U_LT2.hdf",
    "C": "results_Kleiner_C.hdf",
}

KLEINER_COMPACT_PEPTIDE_TABLE_FILES = {
    "P": "kleiner2017_P_peptide_xic.parquet",
    "U": "kleiner2017_U_peptide_xic.parquet",
    "C": "kleiner2017_C_peptide_xic.parquet",
}

KLEINER_FASTA_FILES = {
    "P": FASTA_DIR / "Mock_Comm_RefDB_V3_P.fasta",
    "U": FASTA_DIR / "Mock_Comm_RefDB_V3.fasta",
    "C": FASTA_DIR / "Mock_Comm_RefDB_V3_P.fasta",
}

KLEINER_PEPTIDE_HDF_KEY = "combined_peptide_fdr"
KLEINER_FDR_THRESHOLD = 0.01
KLEINER_OVERWRITE_COMPACT_PEPTIDE_TABLES = False
KLEINER_ALPHAPEPT_HDF_SEARCH_DIRS = [ALPHAPEPT_HDF_DIR, PEPTIDE_TABLE_DIR]

KLEINER_EVIDENCE_PALETTES = {
    "P": ("tab:blue", "#8fb4db"),
    "U": ("tab:orange", "#fdb368"),
    "C": ("tab:green", "#80c680"),
}


## 3. Plotting functions

In [ ]:
def plot_bar_chart(
    data_dict, title,
    ref=None,                             # float OR dict/Series/array
    dash_props=dict(color='0.7', linewidth=1, linestyle='--'),
    sd_dict=None,
    ref_marker_kwargs=None,
    ad_eps=1e-12,
    ad_normalize=True,
    ytick_step=0.05,
    ymax_tick=0.50,
    figsize=(5, 3),
    dpi=300,
    # font sizes
    fs_title: int | float | None = None,
    fs_metrics: int | float = 8,
    fs_ticks: int | float | None = None,
    # metrics box placement/appearance
    metrics_loc: str | None = "upper right",
    metrics_xy: tuple[float, float] | None = None,
    metrics_coords: str = "axes fraction",
    metrics_ha: str | None = None,
    metrics_va: str | None = None,
    metrics_box_kw: dict | None = None,
    # bar color
    bar_color: str = "#5799C7",
    # formatting & embedding
    ad_decimals: int = 3,          # digits for Aitchison in the box
    ax=None,                       # draw into an existing Axes if given
    show: bool = True,             # call plt.show() when we create our own fig
    tight: bool = True,            # call tight_layout() when we create our own fig
):
    """Draw a single panel. If `ax` is provided, draw into it and don't show."""
    organisms = list(data_dict.keys())
    values    = np.array([data_dict[o] for o in organisms], dtype=float)

    yerr = None
    if sd_dict is not None:
        yerr = np.array([sd_dict.get(org, 0.0) for org in organisms], dtype=float)

    def _align_ref_vec(ref_in):
        if ref_in is None:
            return None
        if np.isscalar(ref_in):
            return np.full(len(organisms), float(ref_in), dtype=float)
        if isinstance(ref_in, pd.Series):
            return ref_in.reindex(organisms).astype(float).to_numpy()
        if isinstance(ref_in, dict):
            return np.array([float(ref_in[o]) for o in organisms], dtype=float)
        arr = np.asarray(ref_in, dtype=float)
        if arr.shape[0] != len(organisms):
            raise ValueError("Vector 'ref' length must match number of organisms.")
        return arr

    ref_vec = _align_ref_vec(ref)

    def _aitchison_distance(p, q):
        p = np.asarray(p, float); q = np.asarray(q, float)
        p = np.clip(p, ad_eps, None); q = np.clip(q, ad_eps, None)
        p /= p.sum(); q /= q.sum()
        clr = lambda x: np.log(x) - np.mean(np.log(x))
        diff = clr(p) - clr(q)
        if ad_normalize:
            diff = diff / np.sqrt(len(p))
        return np.linalg.norm(diff)

    if ref_vec is None:
        rmse = mae = ad = np.nan
    else:
        mask = np.isfinite(values) & np.isfinite(ref_vec)
        v = values[mask]; r = ref_vec[mask]
        rmse = float(np.sqrt(np.mean((v - r) ** 2)))
        mae  = float(np.mean(np.abs(v - r)))
        ad   = float(_aitchison_distance(v, r))

    created_here = False
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        created_here = True
    else:
        fig = ax.figure

    # bars
    x = np.arange(len(organisms))
    ax.bar(x, values, yerr=yerr, capsize=3, color=bar_color)

    # set tick positions and labels
    try:
        ax.set_xticks(x, labels=organisms)          # Matplotlib ≥3.6
    except TypeError:
        ax.set_xticks(x)
        ax.set_xticklabels(organisms)               # older versions

    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')


    if fs_title is not None:
        ax.set_title(title, fontsize=fs_title)
    else:
        ax.set_title(title)

    # reference
    if ref is None:
        pass
    elif np.isscalar(ref):
        ax.axhline(float(ref), **dash_props)
    else:
        mk = dict(marker='D', color='0.3', s=5, zorder=3, label='reference')
        if ref_marker_kwargs:
            mk.update(ref_marker_kwargs)
        ax.scatter(x, ref_vec, **mk)

    # metrics text (Aitchison uses ad_decimals)
    txt = f"RMSE = {rmse:.4f}"
    if np.isfinite(mae):
        txt += f"\nMAE = {mae:.4f}\nAitchison = {ad:.{ad_decimals}f}"

    LOCS = {
        "upper right":  ((0.98, 0.98), "right",  "top"),
        "upper left":   ((0.02, 0.98), "left",   "top"),
        "lower right":  ((0.98, 0.02), "right",  "bottom"),
        "lower left":   ((0.02, 0.02), "left",   "bottom"),
        "upper center": ((0.50, 0.98), "center", "top"),
        "lower center": ((0.50, 0.02), "center", "bottom"),
        "center right": ((0.98, 0.50), "right",  "center"),
        "center left":  ((0.02, 0.50), "left",   "center"),
        "center":       ((0.50, 0.50), "center", "center"),
    }

    if metrics_xy is not None:
        xy = metrics_xy
        transform = {"axes fraction": ax.transAxes,
                     "data": ax.transData,
                     "figure fraction": fig.transFigure}[metrics_coords]
        ha = metrics_ha if metrics_ha is not None else "right"
        va = metrics_va if metrics_va is not None else "top"
    elif isinstance(metrics_loc, str):
        if metrics_loc not in LOCS:
            raise ValueError(f"Unknown metrics_loc: {metrics_loc}")
        (xy, def_ha, def_va) = LOCS[metrics_loc]
        transform = ax.transAxes
        ha = metrics_ha if metrics_ha is not None else def_ha
        va = metrics_va if metrics_va is not None else def_va
    else:
        xy = (0.98, 0.95); transform = ax.transAxes
        ha = metrics_ha if metrics_ha is not None else "right"
        va = metrics_va if metrics_va is not None else "top"

    bbox_kw = dict(facecolor='white', edgecolor='0.8', alpha=0.85, boxstyle='round,pad=0.25')
    if metrics_box_kw:
        bbox_kw.update(metrics_box_kw)

    ax.text(xy[0], xy[1], txt, transform=transform, ha=ha, va=va,
            bbox=bbox_kw, fontsize=fs_metrics)

    # y-axis
    ax.set_ylim(0.0, ymax_tick)
    ax.yaxis.set_major_locator(MultipleLocator(ytick_step))
    def _strip_leading_zero(val, pos):
        s = f"{val:.2f}"
        if s in ("0.00", "-0.00"): return "0"
        if s.startswith("-0."):     return "-" + s[2:]
        if s.startswith("0."):      return s[1:]
        return s
    ax.yaxis.set_major_formatter(FuncFormatter(_strip_leading_zero))

    if fs_ticks is not None:
        ax.tick_params(axis='both', which='both', labelsize=fs_ticks)

    if created_here:
        if tight:
            plt.tight_layout()
        if show:
            plt.show()

    return fig, ax, {"rmse": rmse, "mae": mae, "aitchison": ad}

In [ ]:
def _prep_med_sd(df, desired_order, *, lt2_div=None, normalize_cols=True):
    x = df.copy()
    if lt2_div is not None and "LT2" in x.index:
        x.loc["LT2"] = x.loc["LT2"] / float(lt2_div)
    if normalize_cols:
        x = x.div(x.sum(axis=0), axis=1)
    med = x.median(axis=1).reindex(desired_order).dropna()
    sd  = x.std(axis=1, ddof=1).reindex(med.index).fillna(0.0)
    return med.to_dict(), sd.to_dict()

def plot_methods_grid(
    data_by_method: dict[str, dict[str, pd.DataFrame]],
    desired_order: list[str],
    *,
    conditions=("P","U","C"),
    ref_by_cond: dict | None = None,
    lt2_div_by_cond: dict | None = None,
    cond_overrides: dict | None = None,   # {"P": {"ytick_step":0.05, "ymax_tick":0.17}, ...}
    panel_size=(3.0, 2.5), dpi=400,
    row_gap=0.08, col_gap=0.18,
    fs_title=12, fs_metrics=8, fs_ticks=6,
    bar_color="#5799C7",
    metrics_loc="upper left",
    metrics_box_kw=dict(facecolor='white', edgecolor='none', alpha=0),
    ad_decimals=3,
):
    ref_by_cond     = {} if ref_by_cond     is None else ref_by_cond
    lt2_div_by_cond = {} if lt2_div_by_cond is None else lt2_div_by_cond
    cond_overrides  = {} if cond_overrides  is None else cond_overrides

    methods = list(data_by_method.keys())
    n_rows, n_cols = len(methods), len(conditions)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(panel_size[0]*n_cols, panel_size[1]*n_rows),
        dpi=dpi, squeeze=False,
        sharex=False, sharey=False,                 # allow per-condition y settings
        gridspec_kw={'hspace': row_gap, 'wspace': col_gap}
    )

    for i, method in enumerate(methods):
        for j, cond in enumerate(conditions):
            ax = axes[i, j]
            df = data_by_method[method].get(cond, None)
            if df is None or df.empty:
                ax.axis("off")
                continue

            med_dict, sd_dict = _prep_med_sd(
                df, desired_order,
                lt2_div=lt2_div_by_cond.get(cond), normalize_cols=True
            )

            # base kwargs + per-condition overrides
            panel_kwargs = dict(
                fs_title=fs_title, fs_metrics=fs_metrics, fs_ticks=fs_ticks,
                bar_color=bar_color, metrics_loc=metrics_loc, metrics_box_kw=metrics_box_kw,
                ad_decimals=ad_decimals,
            )
            panel_kwargs.update(cond_overrides.get(cond, {}))

            plot_bar_chart(
                med_dict, title=f"{method}: {cond}",
                ref=ref_by_cond.get(cond), sd_dict=sd_dict,
                ax=ax, show=False, tight=False,
                **panel_kwargs
            )

            # make sure y‑tick labels are visible on all columns
            ax.tick_params(axis='y', labelleft=True)

    return fig, axes

In [ ]:
def plot_err_vs_P(
    P_obs: pd.Series,
    err: pd.Series,
    *,
    title,
    top_n_labels: int = 25,
    label_taxa: tuple | list | set = (),
    annotate_kwargs: dict = None,
    figsize=(4,4), dpi=300,
    xmin=-5.5, xmax=2.5,
    ymin=0, ymax=0.13,
    top_k_by_err: int = 20,   # k points with the *smallest* |err|
    # placement & styling
    y_violin: float = 0.1,
    violin_width: float | None = None,
    box_width: float | None = None,
    violin_frac: float = 0.18,
    box_frac: float = 0.12,
    violin_alpha: float = 0.40,
    box_alpha: float = 0.60,
    point_size: float = 18,
    point_alpha: float = 0.9,
    y_label: str = 'P_obs',
    x_label: str | None = 'logFC error',  # set None to hide
    # font-size knobs
    fs_title: float | int | None = None,
    fs_metrics: float | int = 8,
    fs_point_labels: float | int = 7,
    fs_tick_labels: float | int | None = None,
    fs_axis_labels: float | int | None = None,
):
    df = pd.concat({'P_obs': P_obs, 'err': err}, axis=1).dropna()
    x = df['err'].to_numpy()
    y = df['P_obs'].to_numpy()
    abs_err = df['err'].abs()

    # overall metrics
    rmse = float(np.sqrt(np.mean(x**2)))
    bias = float(np.median(x))
    mad  = float(np.median(np.abs(x - bias)))
    sigma_mad = 1.4826 * mad
    q25, q75 = np.percentile(x, [25, 75])
    sigma_iqr = float((q75 - q25) / 1.349)

    # RMSE on k *smallest* |err|
    k = min(top_k_by_err, len(df))
    if k > 0:
        idx_k_small = abs_err.nsmallest(k).index
        x_k = df.loc[idx_k_small, 'err'].to_numpy()
        rmse_topk = float(np.sqrt(np.mean(x_k**2))) if x_k.size else np.nan
    else:
        rmse_topk = np.nan

    # plot
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    # (1) points
    ax.scatter(x, y, s=point_size, alpha=point_alpha, zorder=3)
    ax.axvline(0, color='0.8', ls='--', lw=1, zorder=2)

    # Axes limits first (needed to compute fractional widths)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    # X ticks every 2 units: …, -4, -2, 0, 2, …
    ax.xaxis.set_major_locator(MultipleLocator(2.0))
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.0f'))

    # Compute widths (absolute if provided, else as fraction of y-span)
    yspan = ymax - ymin
    vw = violin_width if violin_width is not None else violin_frac * yspan
    bw = box_width    if box_width    is not None else box_frac    * yspan

    # (2) horizontal violin for err at y=y_violin
    parts = ax.violinplot([x], positions=[y_violin], vert=False,
                          widths=vw, showmeans=False,
                          showmedians=False, showextrema=False)
    for body in parts['bodies']:
        body.set_alpha(violin_alpha)
        body.set_zorder(1)
        body.set_linewidth(1.0)

    # (3) horizontal box at y=y_violin
    bp = ax.boxplot(
        x, vert=False, positions=[y_violin], widths=bw, patch_artist=True,
        showfliers=False,
        boxprops=dict(linewidth=1.2),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        medianprops=dict(linewidth=1.6)
    )
    for patch in bp['boxes']:
        patch.set_alpha(box_alpha)
        patch.set_zorder(2)
    for key in ('medians', 'whiskers', 'caps'):
        for line in bp[key]:
            line.set_zorder(2)

    # Labels & ticks
    if fs_axis_labels is not None:
        if x_label is not None:
            ax.set_xlabel(x_label, fontsize=fs_axis_labels)
        ax.set_ylabel(y_label, fontsize=fs_axis_labels)
    else:
        if x_label is not None:
            ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)

    if fs_title is not None:
        ax.set_title(title, fontsize=fs_title)
    else:
        ax.set_title(title)

    # Y-axis ticks: every 0.05, drop leading zero (".05")
    def _strip_leading_zero(val, pos):
        s = f"{val:.2f}"
        if s in ("0.00", "-0.00"): return "0"
        if s.startswith("-0."):     return "-" + s[2:]  # "-.05"
        if s.startswith("0."):      return s[1:]        # ".05"
        return s
    ax.yaxis.set_major_locator(MultipleLocator(0.05))
    ax.yaxis.set_major_formatter(FuncFormatter(_strip_leading_zero))
    ax.tick_params(axis='y', which='both', labelleft=True)

    # Tick label font size (both axes) if requested
    if fs_tick_labels is not None:
        ax.tick_params(axis='both', which='both', labelsize=fs_tick_labels)

    # metrics textbox
    txt = (
            # f"RMSE = {rmse:.3f}\n"
            # rf"$\mathrm{{RMSE}}_{{\neg \mathrm{{BXL}}}}$" f" = {rmse_topk:.3f}\n"
            rf"$\mathrm{{RMSE}}_{{\mathrm{{top}}20}}$" f" = {rmse_topk:.3f}\n"
        #    f"bias = {bias:.3f} (median)\n"
           f"bias = {bias:.3f}\n"
        #    f"$\\hat\\sigma_{{\\rm MAD}}$ = {sigma_mad:.3f}\n"
           f"$\\hat\\sigma_{{\\rm IQR}}$ = {sigma_iqr:.3f}")

    ax.text(0.02, 0.98, txt, transform=ax.transAxes,
            ha='left', va='top',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.0),
            fontsize=fs_metrics)

    # annotations
    ak = dict(xytext=(3, 3), textcoords='offset points', fontsize=fs_point_labels)
    if annotate_kwargs:
        ak.update(annotate_kwargs)

    if top_n_labels and top_n_labels > 0:
        to_annotate = df.reindex(abs_err.sort_values(ascending=False).head(top_n_labels).index)
        for label, row in to_annotate.iterrows():
            ax.annotate(str(label), (row['err'], row['P_obs']), **ak)

    if label_taxa:
        label_taxa = set(map(str, label_taxa))
        sel = df.index[df.index.map(str).isin(label_taxa)]
        for label in sel:
            row = df.loc[label]
            ax.annotate(str(label), (row['err'], row['P_obs']), **ak)

    plt.tight_layout()
    plt.show()

    return {
        "rmse": rmse,
        "rmse_topk": rmse_topk,
        "k_top": k,
        "bias": bias,
        "sigma_mad": sigma_mad,
        "sigma_iqr": sigma_iqr,
        "n": len(df),
    }

In [ ]:
def plot_delta_logfc_sources_markers(
    datasets: dict[str, pd.DataFrame],     # {"Proteins": df, "Uniques": df, ...}; cols: ["organism","fold_change"] (>0)
    expected: dict,                        # {organism: expected_FC_linear (>0)}
    title: str,
    *,
    desired_order: list,                   # x-axis order (organisms)
    # y-axis (ΔlogFC)
    log_base: int = 2,
    y_min: float = -3.0,
    y_max: float =  3.0,
    # per-point labels
    annotate: bool = True,
    label_mode: str = "delta",             # "delta" | "ratio" | "percent"
    top_label_y: float = 0.98,
    # colors & markers
    colors: dict[str, str] | None = None,
    marker_size: float = 70.0,
    marker_edgecolor: str = "k",
    marker_edgewidth: float = 0.8,
    offset_span: float = 0.50,
    marker_zorder: int = 3,
    # figure & fonts
    figsize: tuple[float, float] = (10, 4),
    title_fs: int = 12,
    axis_label_fs: int = 10,
    tick_fs: int = 9,
    anno_fs: int = 7,
    xtick_rotation: float = 0.0,
    # stats legend/table
    rmse_box_decimals: int = 2,
    rmse_box_title: str | None = None,     # keep None for no header
    rmse_box_fs: int = 9,                  # controls size via FontProperties
    rmse_box_title_fs: int | None = None,
    rmse_box_loc: str = "lower left",
    rmse_box_anchor: tuple[float, float] | None = (0.01, 0.01),  # in-axes coords
    rmse_box_facecolor: str = "white",
    rmse_box_edgecolor: str = "0.6",
    rmse_box_alpha: float = 0.95,
    rmse_box_cols: int | None = None,
    legend_include_bias_sigma: bool = False,
    legend_as_table: bool = False,         # monospaced, aligned columns
    legend_hide_frame: bool = False,       # remove legend frame (border+face)
    legend_sigma_math: bool = True,        # render σ with IQR subscript
    show_n_in_legend: bool = False,
    # console output
    print_stats: bool = True
):
    """
    Plot Δlog_b fold change per taxon for each source and compute per-source errors
    against expected fold changes, all in log space.

    delta = log_b(median_observed_FC) - log_b(expected_FC) = log_b(observed/expected)
    Stats over delta: RMSE, median bias, sigma_IQR (IQR / 1.349)
    """
    # palette
    palette = colors or {
        "Proteins": "#1f77b4",
        "Uniques":  "#ff7f0e",
        "Uniform":  "#2ca02c",
        "Clades":   "#d62728",
        "NMF":      "#9467bd",
        "St.LFQ":   "#09D0EF",
        "TaxonLFQ": "#09D0EF",
    }

    # x-axis order present in data
    present = set().union(*(set(df["organism"].unique()) for df in datasets.values()))
    xorder = [t for t in desired_order if t in present]
    xs = np.arange(len(xorder), dtype=float)

    # axes
    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.set_ylim(y_min, y_max)
    ax.yaxis.set_major_locator(AutoLocator())
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:g}"))
    ax.grid(True, which="both", axis="y", linestyle=":", linewidth=0.6)
    ax.set_ylabel(f"Δlog{log_base} fold change", fontsize=axis_label_fs)
    ax.set_title(title, fontsize=title_fs)
    ax.set_xticks(xs)
    ax.set_xticklabels(xorder, fontsize=tick_fs, rotation=xtick_rotation)
    ax.tick_params(axis="y", labelsize=tick_fs)

    # zero reference per taxon
    for i, _t in enumerate(xorder):
        ax.hlines(0.0, i - 0.35, i + 0.35, linestyles="dashed", linewidth=1.2, color="0.25")

    # offsets for series
    keys = list(datasets.keys())
    shifts = np.linspace(-offset_span/2.0, +offset_span/2.0, len(keys)) if len(keys) > 1 else np.array([0.0])

    # transforms
    top_tf = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)

    # helpers
    ln_base = np.log(log_base)
    def log_b(x: float) -> float:
        return np.log(x) / ln_base

    # stats
    stats_rows = []
    legend_stats = {}

    for j, (key, df) in enumerate(datasets.items()):
        col = palette.get(key, f"C{j}")
        shift = shifts[j]

        med = (df.groupby("organism")["fold_change"]
                 .median()
                 .replace([np.inf, -np.inf], np.nan)
                 .dropna())

        xs_draw, ys_delta, deltas = [], [], []

        for i, t in enumerate(xorder):
            if t not in med.index:
                continue
            exp = expected.get(t, None)
            if exp is None:
                continue

            obs_fc = float(med.loc[t])
            exp_fc = float(exp)
            if not (np.isfinite(obs_fc) and np.isfinite(exp_fc) and obs_fc > 0 and exp_fc > 0):
                continue

            dlt = log_b(obs_fc) - log_b(exp_fc)
            xs_draw.append(xs[i] + shift)
            ys_delta.append(dlt)
            deltas.append(dlt)

            if annotate:
                if label_mode == "delta":
                    text = f"{dlt:+.2f}"
                elif label_mode == "ratio":
                    text = f"×{(log_base ** dlt):.2f}"
                elif label_mode == "percent":
                    text = f"{((log_base ** dlt) - 1.0)*100:+.0f}%"
                else:
                    text = f"{dlt:+.2f}"
                ax.text(xs[i] + shift, top_label_y, text,
                        transform=top_tf, rotation=90,
                        ha="center", va="top", fontsize=anno_fs, color=col)

        if xs_draw:
            ax.scatter(np.array(xs_draw), np.array(ys_delta),
                       c=col, edgecolors=marker_edgecolor,
                       s=marker_size, zorder=marker_zorder,
                       linewidths=marker_edgewidth, marker="D")

            d = np.array(deltas, dtype=float)
            rmse = float(np.sqrt(np.mean(d**2)))
            bias = float(np.median(d))
            if d.size >= 2:
                q1, q3 = np.percentile(d, [25, 75])
                sigma_iqr = float((q3 - q1) / 1.349)
            else:
                sigma_iqr = np.nan

            legend_stats[key] = (rmse, bias, sigma_iqr, len(d))
            stats_rows.append({"source": key, "n": len(d),
                               "rmse": rmse, "bias": bias, "sigma_IQR": sigma_iqr})

    stats_df = pd.DataFrame(stats_rows).set_index("source") if stats_rows else pd.DataFrame()

    # legend/table
    if legend_stats:
        handles, labels = [], []

        # alignment helpers
        name_w = max(len(k) for k in legend_stats.keys()) if legend_as_table else 0
        num_w = rmse_box_decimals + 4  # width for numbers like ' -0.12'
        def fmt_num(x, signed=False):
            if x is None or (isinstance(x, float) and (math.isnan(x) or not np.isfinite(x))):
                return " " * num_w
            return (f"{x:+.{rmse_box_decimals}f}" if signed else f"{x:.{rmse_box_decimals}f}").rjust(num_w)

        sigma_label = r"$\sigma_{\mathrm{IQR}}$" if legend_sigma_math else "s_IQR"

        for j, key in enumerate(keys):
            if key not in legend_stats:
                continue
            col = palette.get(key, f"C{j}")
            handles.append(Line2D([], [], marker="D", linestyle="None",
                                  markersize=8, markeredgecolor=marker_edgecolor,
                                  markeredgewidth=marker_edgewidth, color=col))
            rmse, bias, sigma_iqr, n_val = legend_stats[key]

            if legend_include_bias_sigma:
                # build one row; keep numbers aligned by padding in a monospace font
                left = f"{key:<{name_w}}  " if legend_as_table else f"{key}  "
                row = (
                    f"{left}"
                    f"RMSE {fmt_num(rmse, signed=False)}  "
                    f"bias {fmt_num(bias, signed=True)}  "
                    f"{sigma_label} {fmt_num(sigma_iqr, signed=False)}"
                )
                if show_n_in_legend:
                    row += f"  n={n_val:d}"
            else:
                left = f"{key:<{name_w}}  " if legend_as_table else f"{key}  "
                row = f"{left}RMSE {fmt_num(rmse)}"
                if show_n_in_legend:
                    row += f"  n={n_val:d}"

            labels.append(row)

        # Build a single FontProperties object with family+size so size always applies
        if legend_as_table:
            font_props = FontProperties(family="monospace", size=rmse_box_fs)
        else:
            font_props = FontProperties(size=rmse_box_fs)

        kwargs = dict(loc=rmse_box_loc, frameon=True, fancybox=True,
                      framealpha=rmse_box_alpha, title=rmse_box_title)
        if rmse_box_anchor is not None:
            kwargs["bbox_to_anchor"] = rmse_box_anchor
            kwargs["bbox_transform"] = ax.transAxes
        if rmse_box_cols is not None:
            kwargs["ncol"] = rmse_box_cols

        leg = ax.legend(handles=handles, labels=labels, prop=font_props, **kwargs)

        if rmse_box_title_fs is not None and rmse_box_title:
            try:
                leg.set_title(rmse_box_title, prop={"size": rmse_box_title_fs})
            except Exception:
                pass

        frame = leg.get_frame()
        if legend_hide_frame:
            frame.set_visible(False)
        else:
            frame.set_facecolor(rmse_box_facecolor)
            frame.set_edgecolor(rmse_box_edgecolor)

    if print_stats and not stats_df.empty:
        with pd.option_context("display.float_format", "{:.3f}".format):
            print("\nFold-change summary per source (Δlog units):")
            print(stats_df.loc[:, ["n", "rmse", "bias", "sigma_IQR"]])

    plt.tight_layout()
    plt.show()
    return fig, ax, stats_df

In [ ]:
def _kleiner_condition_legend_handles():
    handles, labels = [], []
    for cond in KLEINER_CONDITIONS:
        unique_col, shared_col = KLEINER_EVIDENCE_PALETTES[cond]
        h_unique = Rectangle((0, 0), 1, 1, facecolor=unique_col, edgecolor="none")
        h_shared = Rectangle((0, 0), 1, 1, facecolor=shared_col, edgecolor="none")
        handles.append((h_unique, h_shared))
        labels.append(cond)
    return handles, labels


def add_kleiner_condition_legend(
    ax,
    *,
    title="Conditions (unique peptides | shared peptides):",
    ncol=3,
    y_above_axes=1.18,
):
    handles, labels = _kleiner_condition_legend_handles()
    fig = ax.figure
    title_fs = ax.title.get_size()

    leg = fig.legend(
        handles,
        labels,
        title=title,
        loc="upper center",
        bbox_to_anchor=(0.5, y_above_axes),
        ncol=ncol,
        handler_map={tuple: HandlerTuple(ndivide=None)},
        markerfirst=False,
        handlelength=2.2,
        handleheight=1.1,
        handletextpad=0.6,
        columnspacing=1.6,
        frameon=True,
        fancybox=False,
        edgecolor="0.3",
        prop={"size": title_fs},
    )
    leg.get_title().set_fontsize(title_fs)
    frame = leg.get_frame()
    frame.set_linewidth(0.8)
    frame.set_facecolor("white")
    frame.set_alpha(1.0)
    leg.set_in_layout(True)
    return leg


def plot_kleiner_condition_legend(figsize=(5.0, 0.7), dpi=300):
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    ax.axis("off")
    add_kleiner_condition_legend(ax, y_above_axes=0.9)
    return fig, ax


def plot_kleiner_intensity_fractions(
    fractions,
    *,
    title="Fractions of total intensity",
    figsize=(3, 3),
    dpi=300,
    fs_title=11,
    fs_ticks=8,
    fs_annot=7,
):
    cats = ["unique peptides", "shared peptides"]
    x = np.arange(len(cats))
    width = 0.315
    offsets = (-width, 0.0, width)

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    for i, cond in enumerate(KLEINER_CONDITIONS):
        unique_col, shared_col = KLEINER_EVIDENCE_PALETTES[cond]
        vals = [fractions[cond]["unique"] * 100.0, fractions[cond]["shared"] * 100.0]
        bars = ax.bar(
            x + offsets[i],
            vals,
            width,
            color=[unique_col, shared_col],
            edgecolor="none",
        )
        for bar, val in zip(bars, vals):
            ax.annotate(
                f"{val:.1f}%",
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=fs_annot,
            )

    ax.set_xticks(x, cats)
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_formatter(PercentFormatter(100, decimals=0))
    ax.set_title(title, pad=10, fontsize=fs_title)
    ax.tick_params(axis="both", which="both", labelsize=fs_ticks)
    fig.tight_layout()
    return fig, ax


def plot_kleiner_distinct_counts(
    counts,
    *,
    title="Counts of distinct peptides",
    group_sep=0.75,
    y_top_count=None,
    figsize=(3, 3),
    dpi=300,
    fs_title=11,
    fs_ticks=8,
    fs_annot=6.2,
):
    cats = ["unique peptides", "shared peptides"]
    x = np.array([0.0, float(group_sep)])
    width = 0.22
    offsets = (-width, 0.0, width)

    if y_top_count is None:
        y_top_count = max(
            counts[cond][kind]
            for cond in KLEINER_CONDITIONS
            for kind in ["unique", "shared"]
        )
        y_top_count = int(math.ceil(y_top_count / 20_000) * 20_000)

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    for i, cond in enumerate(KLEINER_CONDITIONS):
        unique_col, shared_col = KLEINER_EVIDENCE_PALETTES[cond]
        vals = [counts[cond]["unique"], counts[cond]["shared"]]
        bars = ax.bar(
            x + offsets[i],
            vals,
            width,
            color=[unique_col, shared_col],
            edgecolor="none",
        )
        total = sum(vals)
        for bar, val in zip(bars, vals):
            pct = 100.0 * val / total if total else 0.0
            ax.annotate(
                f"{val:,}\n({pct:.1f}%)",
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=fs_annot,
            )

    ax.set_ylim(0, y_top_count)
    ax.yaxis.set_major_locator(MultipleLocator(20_000))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: "" if v <= 0 else f"{int(v / 1000)}k"))
    ax.set_xticks(x, cats)
    ax.set_title(title, pad=10, fontsize=fs_title)
    ax.tick_params(axis="both", which="both", labelsize=fs_ticks)
    ax.margins(y=0.1)
    fig.tight_layout()
    return fig, ax


def bucket_shared(counter, kmin=2, kmax=5):
    bins = [counter.get(k, 0) for k in range(kmin, kmax + 1)]
    bins.append(sum(v for k, v in counter.items() if k > kmax))
    return np.asarray(bins, dtype=int)


def plot_kleiner_peptide_sharing(
    sharing_counts,
    *,
    kmin=2,
    kmax=5,
    title="Peptide sharing",
    group_width=0.9,
    figsize=(3, 3),
    dpi=300,
    fs_title=11,
    fs_ticks=8,
    fs_xlabel=8,
):
    labels = [str(k) for k in range(kmin, kmax + 1)] + [f"{kmax + 1}+"]
    data = np.vstack([bucket_shared(sharing_counts[cond], kmin, kmax) for cond in KLEINER_CONDITIONS])

    x = np.arange(len(labels))
    n = len(KLEINER_CONDITIONS)
    width = group_width / n
    offsets = (np.arange(n) - (n - 1) / 2) * width

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    for i, cond in enumerate(KLEINER_CONDITIONS):
        _, shared_col = KLEINER_EVIDENCE_PALETTES[cond]
        ax.bar(x + offsets[i], data[i], width, label=cond, color=shared_col)

    ax.set_xticks(x, labels)
    ax.set_xlabel("number of organisms a peptide maps to", fontsize=fs_xlabel)
    ax.set_title(title, pad=10, fontsize=fs_title)
    ax.tick_params(axis="both", which="both", labelsize=fs_ticks)

    y_top = int(math.ceil(data.max() / 2000.0) * 2000) or 2000
    ax.set_ylim(0, y_top)
    ax.yaxis.set_major_locator(MultipleLocator(2000))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: "" if v <= 0 else f"{int(v / 1000)}k"))
    ax.margins(y=0.1)
    fig.tight_layout()
    return fig, ax


def trunc_label(label, max_len=20):
    label = str(label)
    return label[:max_len] + "…" if len(label) > max_len else label


def plot_kleiner_taxon_counts(
    count_table,
    ref_series,
    *,
    condition,
    order=None,
    title=" ",
    figsize=(10, 3),
    dpi=300,
    y_top_count=13_000,
    y2_top=0.35,
    show_expected_legend=True,
    fs_ticks=8,
):
    if order is None:
        order = KLEINER_ORDER_COMPLETE[condition]

    df = count_table.reindex(order).fillna(0.0)
    x_labels = [trunc_label(x) for x in df.index]
    x = np.arange(len(x_labels))

    unique_col, shared_col = KLEINER_EVIDENCE_PALETTES[condition]

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    ax.bar(x, df["Unique"].astype(float), label="unique peptides", color=unique_col)
    ax.bar(
        x,
        df["NonUnique"].astype(float),
        bottom=df["Unique"].astype(float),
        color=shared_col,
        label="shared peptides",
    )

    ax.set_ylim(0, y_top_count)
    ax.yaxis.set_major_locator(MultipleLocator(2000))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: "" if v <= 0 else f"{int(v / 1000)}k"))
    ax.set_ylabel("peptide count")
    ax.set_title(title, loc="left")
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=fs_ticks)
    ax.tick_params(axis="y", labelsize=fs_ticks)

    ax2 = ax.twinx()
    y2 = ref_series.reindex(order).fillna(0.0).astype(float)
    ax2.scatter(
        x,
        y2.values,
        marker="*",
        color="black",
        s=50,
        zorder=3,
        label="expected proteinaceous biomass",
    )
    ax2.set_ylabel("proteinaceous biomass fraction")
    ax2.tick_params(axis="y", colors="black", labelsize=fs_ticks)
    ax2.spines["right"].set_color("black")
    ax2.set_ylim(0, y2_top)
    ax2.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:.2f}"))
    ax2.yaxis.set_major_locator(MultipleLocator(0.10))

    if show_expected_legend:
        handles, labels = ax2.get_legend_handles_labels()
        ax.legend(handles, labels, frameon=False, loc="upper right")

    fig.tight_layout()
    return fig, (ax, ax2)


## 4. Preprocessing helpers

In [ ]:
def compositional_logFC(
    numer: pd.DataFrame,
    denom: pd.DataFrame,
    *,
    order: list | None = None,     # optional explicit taxon order
    eps: float = 1e-12,            # zero guard for logs/closure
    return_parts: bool = False,    # also return numer_obs, denom_obs, etc.
) -> pd.Series | dict:
    """
    Compute log2 fold-change between two conditions in *composition space*.

    Steps:
      1) Close each run to a composition (columns sum to 1).
      2) Replace zeros by `eps` and re-close (per run).
      3) Center across runs with the component-wise median (per taxon).
      4) Re-close the median vectors to valid compositions.
      5) Return log2(numer_obs) - log2(denom_obs).

    Parameters
    ----------
    numer, denom : DataFrame
        Rows = taxa, columns = runs (replicates). Non-negative values.
    order : list, optional
        If provided, taxa will be reindexed to this order (intersection is used).
        Otherwise taxa are aligned by the intersection in the order of `numer`.
    eps : float
        Small positive constant for zero-guarding before logs.
    return_parts : bool
        If True, return a dict with logFC_hat and intermediates.

    Returns
    -------
    logFC_hat : Series (if return_parts=False)
        Index = taxa, values = log2 fold-changes in composition space.
        Or a dict with additional components if return_parts=True.
    """

    def _intersect_in_order(idx_a, idx_b):
        bset = set(idx_b)
        return [i for i in idx_a if i in bset]

    # 0) Align taxa
    if order is not None:
        common = [t for t in order if (t in numer.index) and (t in denom.index)]
    else:
        common = _intersect_in_order(numer.index, denom.index)
    if len(common) == 0:
        raise ValueError("No overlapping taxa between numer and denom (after optional ordering).")

    N = numer.loc[common].astype(float)
    D = denom.loc[common].astype(float)

    # helper: closure per run (column-wise)
    def _close_cols(df):
        col_sums = df.sum(axis=0, skipna=True)
        # avoid /0; columns with sum==0 become NaN
        closed = df.div(col_sums.where(col_sums != 0, np.nan), axis=1)
        return closed

    # 1) Close each run to compositions
    N_comp = _close_cols(N)
    D_comp = _close_cols(D)

    # 2) Zero replacement + re-close
    N_comp = _close_cols(N_comp.clip(lower=eps))
    D_comp = _close_cols(D_comp.clip(lower=eps))

    # 3) Condition centers (component-wise medians across runs)
    N_med = N_comp.median(axis=1, skipna=True)
    D_med = D_comp.median(axis=1, skipna=True)

    # 4) Re-close medians to valid compositions
    numer_obs = N_med / N_med.sum(skipna=True)
    denom_obs = D_med / D_med.sum(skipna=True)

    # 5) Compositional log2 fold-change
    logFC_hat = np.log2(numer_obs.clip(lower=eps)) - np.log2(denom_obs.clip(lower=eps))
    logFC_hat.name = "log2FC_numer_vs_denom"

    if not return_parts:
        return logFC_hat

    return {
        "logFC_hat": logFC_hat,
        "numer_obs": numer_obs,
        "denom_obs": denom_obs,
        "numer_comp": N_comp,   # per-run compositions (columns sum to 1)
        "denom_comp": D_comp,
        "numer_med": N_med,
        "denom_med": D_med,
        "taxa_order": common,
        "eps": eps,
    }

In [ ]:
def read_composition(path: str | Path) -> pd.DataFrame:
    """Load and tidy a composition *.tab file."""
    df = pd.read_csv(path, sep="\t", quotechar='"', encoding="latin1", engine="python")
    cols = (
        pd.Series(df.columns)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    df.columns = cols

    rename_map = {
        "Protein input per biological replicate (mug)": "Protein_µg",
        "Protein input per biological replicate (µg)": "Protein_µg",
        "Protein abundance %": "Protein_%",
        "Cell number input per biological replicate": "Cells",
        "Cell abundance %": "Cell_%",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    num_cols = [c for c in ["Protein_µg", "Protein_%", "Cells", "Cell_%"] if c in df.columns]
    if num_cols:
        df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

    if "Label" in df.columns:
        df = df.set_index("Label")
    return df


def make_ref_from_df(df: pd.DataFrame, desired_order_complete: list[str], value_col: str = "Protein_%") -> pd.Series:
    """Return reference proportions aligned to desired_order_complete, summing LT2a/LT2b/LT2c into LT2."""
    s = df[value_col].copy()
    lt2_parts = [x for x in ["LT2a", "LT2b", "LT2c"] if x in s.index]
    if lt2_parts:
        s["LT2"] = s.get("LT2", 0.0) + s.loc[lt2_parts].sum()
        s = s.drop(lt2_parts)
    return (s / 100.0).reindex(desired_order_complete).dropna()


def read_parquet_matrix(path: str | Path, *, normalize_columns: bool = False) -> pd.DataFrame:
    """Read an organism × run matrix, optionally closing each run to proportions."""
    df = pd.read_parquet(path).apply(pd.to_numeric, errors="coerce")
    if normalize_columns:
        df = df.div(df.sum(axis=0), axis=1)
    return df


def load_method_condition_tables(input_dir: Path, file_map: dict, *, normalize_columns: bool = False) -> dict:
    """Load nested {method: {condition_or_sample: DataFrame}} parquet files from one input folder."""
    return {
        method: {
            condition: read_parquet_matrix(input_dir / filename, normalize_columns=normalize_columns)
            for condition, filename in condition_files.items()
        }
        for method, condition_files in file_map.items()
    }


def normalize_reference_to_order(ref: pd.Series, order: list[str]) -> pd.Series:
    """Subset a reference Series to order and renormalize to sum to one."""
    out = ref.loc[order].astype(float)
    return out / out.sum()


def compute_kleiner_logfc_result(
    method_tables: dict[str, pd.DataFrame],
    ref_props: dict[str, pd.Series],
    *,
    numerator: str,
    denominator: str,
    order: list[str],
) -> dict:
    """Compute observed and expected compositional log2 fold-change for one Kleiner contrast."""
    parts = compositional_logFC(
        method_tables[numerator],
        method_tables[denominator],
        order=order,
        return_parts=True,
    )
    logFC_hat = parts["logFC_hat"]
    logFC_true = np.log2(ref_props[numerator]) - np.log2(ref_props[denominator])
    err = (logFC_hat - logFC_true).dropna()

    return {
        "parts": parts,
        "logFC_hat": logFC_hat,
        "logFC_true": logFC_true,
        "err": err,
        "numer_obs": parts["numer_obs"],
        "denom_obs": parts["denom_obs"],
    }


def foldchange_pairs(A: pd.DataFrame, B: pd.DataFrame) -> pd.DataFrame:
    """Return all run-pair fold changes B/A as a long table."""
    taxa = A.index.intersection(B.index)
    A = A.loc[taxa].replace(0, np.nan)
    B = B.loc[taxa].replace(0, np.nan)

    fc = B.to_numpy()[:, :, None] / A.to_numpy()[:, None, :]
    pairs = [f"{b}/{a}" for b in B.columns for a in A.columns]

    long = (
        pd.DataFrame(fc.reshape(len(taxa), -1), index=taxa, columns=pairs)
        .stack()
        .rename("fold_change")
        .reset_index()
        .rename(columns={"level_0": "organism", "level_1": "pair"})
    )
    return long.replace([np.inf, -np.inf], np.nan).dropna(subset=["fold_change"])


def compute_zhao_foldchanges(sample_tables: dict[str, pd.DataFrame], contrasts: dict) -> dict[str, pd.DataFrame]:
    """Compute all configured Zhao fold-change contrasts for one method."""
    out = {}
    for contrast, cfg in contrasts.items():
        out[contrast] = foldchange_pairs(
            sample_tables[cfg["denominator"]],
            sample_tables[cfg["numerator"]],
        )
    return out


def make_zhao_author_fc(offsets: list[float], expected_fc: dict, order: list[str]) -> pd.DataFrame:
    """Back-calculate author median fold changes from stored offsets and expected fold changes."""
    dist = pd.Series(offsets, index=order, dtype=float)
    exp_series = pd.Series(expected_fc).reindex(order).astype(float)
    median_fc = exp_series + dist
    return pd.DataFrame({
        "organism": order,
        "pair": "authors_median",
        "fold_change": median_fc.values,
    })

# B: Kleiner 2017

In [ ]:
HEADER_RE = re.compile(r"^([^_]+)_")


def fasta_by_prefix(fasta_path, ignore_numeric=False):
    org2seqs = defaultdict(list)

    for rec in SeqIO.parse(str(fasta_path), "fasta"):
        m = HEADER_RE.match(rec.id)
        if not m:
            continue
        prefix = m.group(1)
        if ignore_numeric and prefix.isdigit():
            continue
        org2seqs[prefix].append(str(rec.seq).upper())

    if "CRAP" in org2seqs and "cRAP" not in org2seqs:
        org2seqs["cRAP"] = org2seqs.pop("CRAP")

    return dict(org2seqs)


def build_ac_automaton(peptides):
    if ahocorasick is None:
        raise ImportError("Install pyahocorasick to map compact peptide tables to FASTA sequences.")
    A = ahocorasick.Automaton()
    for idx, pep in enumerate(peptides):
        A.add_word(str(pep), (idx, str(pep)))
    A.make_automaton()
    return A


def map_peptides_ac(peptides, org2seqs):
    peptides = [str(p) for p in peptides]
    idx2pep = dict(enumerate(peptides))
    pep2tax = [set() for _ in peptides]
    A = build_ac_automaton(peptides)

    for taxon, protein_list in org2seqs.items():
        for prot in protein_list:
            for _, (idx, _pep) in A.iter(prot):
                pep2tax[idx].add(taxon)

    return {
        idx2pep[i]: sorted(taxa)
        for i, taxa in enumerate(pep2tax)
        if taxa
    }


def invert_peptide_map(pep2orgs, all_orgs=None, keep_empty=True):
    org_to_peps = defaultdict(list)
    if keep_empty and all_orgs is not None:
        for org in all_orgs:
            org_to_peps[org]

    for pep, orgs in pep2orgs.items():
        for org in orgs:
            org_to_peps[org].append(pep)

    return dict(org_to_peps)


def _decoy_keep_mask(decoy_col):
    if decoy_col.dtype == bool:
        return ~decoy_col
    return decoy_col.astype(str).str.strip().str.lower().isin({"false", "0", "no"})


def load_alphapept_peptides(path, *, key=KLEINER_PEPTIDE_HDF_KEY, fdr_threshold=KLEINER_FDR_THRESHOLD):
    df = pd.read_hdf(path, key=key)
    df = df[df["fdr"] < fdr_threshold].copy()
    if "decoy" in df.columns:
        df = df[_decoy_keep_mask(df["decoy"])].copy()
    df["sequence_naked"] = df["sequence_naked"].astype(str)
    df["ms1_int_sum_area"] = pd.to_numeric(df["ms1_int_sum_area"], errors="coerce")
    return df


def build_xic(df):
    return (
        df.groupby("sequence_naked", as_index=False)
        .agg(ms1_int_sum_area=("ms1_int_sum_area", "sum"))
        .sort_values("sequence_naked")
        .reset_index(drop=True)
    )


def locate_alphapept_hdf(
    cond,
    *,
    hdf_files=KLEINER_ALPHAPEPT_HDF_FILES,
    search_dirs=KLEINER_ALPHAPEPT_HDF_SEARCH_DIRS,
):
    filename = hdf_files[cond]
    for folder in search_dirs:
        path = Path(folder) / filename
        if path.exists():
            return path
    searched = chr(10).join(str(Path(folder) / filename) for folder in search_dirs)
    raise FileNotFoundError(f"Could not find AlphaPept HDF for {cond}. Searched:{chr(10)}{searched}")


def compact_peptide_table_path(
    cond,
    *,
    peptide_table_dir=PEPTIDE_TABLE_DIR,
    table_files=KLEINER_COMPACT_PEPTIDE_TABLE_FILES,
):
    return Path(peptide_table_dir) / table_files[cond]


def write_compact_peptide_table(xic, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    suffixes = "".join(path.suffixes).lower()

    if suffixes.endswith(".parquet"):
        xic.to_parquet(path, index=False)
    elif suffixes.endswith(".tsv") or suffixes.endswith(".tsv.gz"):
        xic.to_csv(path, sep="\t", index=False)
    elif suffixes.endswith(".csv") or suffixes.endswith(".csv.gz"):
        xic.to_csv(path, index=False)
    else:
        raise ValueError(f"Unsupported compact peptide table format: {path}")


def read_compact_peptide_table(path):
    path = Path(path)
    suffixes = "".join(path.suffixes).lower()

    if suffixes.endswith(".parquet"):
        xic = pd.read_parquet(path)
    elif suffixes.endswith(".tsv") or suffixes.endswith(".tsv.gz"):
        xic = pd.read_csv(path, sep="\t")
    elif suffixes.endswith(".csv") or suffixes.endswith(".csv.gz"):
        xic = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported compact peptide table format: {path}")

    xic = xic.loc[:, ["sequence_naked", "ms1_int_sum_area"]].copy()
    xic["sequence_naked"] = xic["sequence_naked"].astype(str)
    xic["ms1_int_sum_area"] = pd.to_numeric(xic["ms1_int_sum_area"], errors="coerce")
    return xic.dropna(subset=["sequence_naked"]).reset_index(drop=True)


def extract_compact_peptide_table_from_hdf(cond, *, overwrite=False):
    out_path = compact_peptide_table_path(cond)
    if out_path.exists() and not overwrite:
        return read_compact_peptide_table(out_path)

    hdf_path = locate_alphapept_hdf(cond)
    df = load_alphapept_peptides(hdf_path)
    xic = build_xic(df)
    write_compact_peptide_table(xic, out_path)
    return xic


def ensure_compact_peptide_tables(conditions=KLEINER_CONDITIONS, *, overwrite=False):
    tables = {}
    for cond in conditions:
        out_path = compact_peptide_table_path(cond)
        if out_path.exists() and not overwrite:
            tables[cond] = read_compact_peptide_table(out_path)
            source = "existing compact table"
        else:
            tables[cond] = extract_compact_peptide_table_from_hdf(cond, overwrite=overwrite)
            source = "AlphaPept HDF extraction"

        size_mb = out_path.stat().st_size / 1e6 if out_path.exists() else np.nan
        print(f"{cond}: {len(tables[cond]):,} peptides from {source} -> {out_path} ({size_mb:.2f} MB)")

    return tables


def standardize_taxon_count_table(df, order=None):
    out = df.copy()
    if "Organism" in out.columns:
        out = out.set_index("Organism")

    out = out.rename(
        columns={
            "unique": "Unique",
            "UniquePeptides": "Unique",
            "shared": "NonUnique",
            "Shared": "NonUnique",
            "non_unique": "NonUnique",
            "Nonunique": "NonUnique",
            "Non-unique": "NonUnique",
        }
    )

    out = out.rename(index={"CRAP": "cRAP"})
    out.index = out.index.astype(str)

    for col in ["Unique", "NonUnique"]:
        if col not in out.columns:
            out[col] = 0.0
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(0.0)

    out = out[["Unique", "NonUnique"]]
    if order is not None:
        out = out.reindex(order).fillna(0.0)

    out["Total"] = out["Unique"] + out["NonUnique"]
    return out


def build_taxon_count_table(pep2orgs, order=None):
    org2peps = invert_peptide_map(pep2orgs)
    org2peps_unique = defaultdict(list)

    for pep, owners in pep2orgs.items():
        if len(owners) == 1:
            org2peps_unique[owners[0]].append(pep)

    rows = []
    for org, all_peps in org2peps.items():
        total = len(all_peps)
        unique = len(org2peps_unique.get(org, []))
        rows.append({"Organism": org, "Unique": unique, "NonUnique": total - unique})

    df = pd.DataFrame(rows).set_index("Organism")
    return standardize_taxon_count_table(df, order)


def summarize_kleiner_condition_from_peptides(
    cond,
    *,
    peptide_tables=None,
    peptide_table_dir=PEPTIDE_TABLE_DIR,
    compact_peptide_table_files=KLEINER_COMPACT_PEPTIDE_TABLE_FILES,
    fasta_files=KLEINER_FASTA_FILES,
    order_by_condition=KLEINER_ORDER_COMPLETE,
):
    if peptide_tables is None:
        xic = read_compact_peptide_table(Path(peptide_table_dir) / compact_peptide_table_files[cond])
    else:
        xic = peptide_tables[cond].copy()

    distinct_peps = pd.Index(xic["sequence_naked"].unique().astype(str))

    org2seqs = fasta_by_prefix(fasta_files[cond])
    pep2orgs = map_peptides_ac(distinct_peps, org2seqs)

    unique_map = {
        pep: orgs
        for pep, orgs in pep2orgs.items()
        if len(orgs) == 1
    }
    unique_peps = set(unique_map)
    mapped_peps = set(pep2orgs)
    non_unique_peps = mapped_peps - unique_peps

    intensity_lookup = dict(zip(xic["sequence_naked"], xic["ms1_int_sum_area"]))
    sum_unique = float(sum(intensity_lookup.get(p, 0.0) for p in unique_peps))
    sum_non_unique = float(sum(intensity_lookup.get(p, 0.0) for p in non_unique_peps))
    denom = sum_unique + sum_non_unique

    class_fractions = {
        "unique": sum_unique / denom if denom else np.nan,
        "shared": sum_non_unique / denom if denom else np.nan,
    }
    class_counts = {
        "unique": len(unique_peps),
        "shared": len(non_unique_peps),
    }
    sharing_counts = Counter(len(orgs) for orgs in pep2orgs.values())
    taxon_count_table = build_taxon_count_table(pep2orgs, order_by_condition[cond])

    return {
        "xic": xic,
        "distinct_peps": distinct_peps,
        "org2seqs": org2seqs,
        "pep2orgs": pep2orgs,
        "unique_peps": unique_peps,
        "non_unique_peps": non_unique_peps,
        "sum_unique": sum_unique,
        "sum_non_unique": sum_non_unique,
        "class_fractions": class_fractions,
        "class_counts": class_counts,
        "sharing_counts": sharing_counts,
        "taxon_count_table": taxon_count_table,
    }


def summarize_all_kleiner_conditions_from_peptides(conditions=KLEINER_CONDITIONS, *, peptide_tables=None):
    summaries = {}
    for cond in conditions:
        summaries[cond] = summarize_kleiner_condition_from_peptides(cond, peptide_tables=peptide_tables)
        s = summaries[cond]
        print(
            f"{cond}: distinct={len(s['distinct_peps']):,}, "
            f"mapped={len(s['pep2orgs']):,}, "
            f"unique={s['class_counts']['unique']:,}, "
            f"shared={s['class_counts']['shared']:,}"
        )

    peptide_class_fractions = {
        cond: summaries[cond]["class_fractions"]
        for cond in conditions
    }
    peptide_class_counts = {
        cond: summaries[cond]["class_counts"]
        for cond in conditions
    }
    peptide_sharing_counts = {
        cond: summaries[cond]["sharing_counts"]
        for cond in conditions
    }
    taxon_count_tables = {
        cond: summaries[cond]["taxon_count_table"]
        for cond in conditions
    }

    return summaries, peptide_class_fractions, peptide_class_counts, peptide_sharing_counts, taxon_count_tables


## 5. Load Kleiner reference compositions

In [ ]:
kleiner_compositions = {
    cond: read_composition(RECIPE_DIR / filename)
    for cond, filename in KLEINER_COMPOSITION_FILES.items()
}
kleiner_compositions["C"] = kleiner_compositions["C"].iloc[:28, :]

for cond, labels in KLEINER_LABELS.items():
    kleiner_compositions[cond].index = labels

kleiner_ref_series = {
    cond: make_ref_from_df(
        kleiner_compositions[cond],
        KLEINER_ORDER_COMPLETE[cond],
        value_col="Protein_%",
    )
    for cond in KLEINER_CONDITIONS
}

kleiner_panel_ref_series = {
    cond: kleiner_ref_series[cond].reindex(KLEINER_ORDER_COMPLETE[cond]).fillna(0.0)
    for cond in KLEINER_CONDITIONS
}

kleiner_ref_props = {
    cond: normalize_reference_to_order(kleiner_ref_series[cond], KLEINER_ORDER)
    for cond in KLEINER_CONDITIONS
}

P_exp = kleiner_ref_props["P"]
U_exp = kleiner_ref_props["U"]
C_exp = kleiner_ref_props["C"]

print("Reference sums:")
print(pd.Series({cond: ref.sum() for cond, ref in kleiner_ref_props.items()}))


## 6. Extract/load compact Kleiner peptide tables

In [ ]:
kleiner_compact_peptide_tables = ensure_compact_peptide_tables(
    KLEINER_CONDITIONS,
    overwrite=KLEINER_OVERWRITE_COMPACT_PEPTIDE_TABLES,
)

kleiner_compact_peptide_table_summary = pd.DataFrame({
    cond: {
        "n_peptides": len(table),
        "total_ms1_intensity": table["ms1_int_sum_area"].sum(),
    }
    for cond, table in kleiner_compact_peptide_tables.items()
}).T

kleiner_compact_peptide_table_summary

## 7. Compute Kleiner peptide-to-taxon evidence summaries

In [ ]:
(
    kleiner_condition_summaries,
    kleiner_peptide_class_fractions,
    kleiner_peptide_class_counts,
    kleiner_peptide_sharing_counts,
    kleiner_taxon_count_tables,
) = summarize_all_kleiner_conditions_from_peptides(
    KLEINER_CONDITIONS,
    peptide_tables=kleiner_compact_peptide_tables,
)

## 8. Plot Kleiner peptide-sharing evidence panels

In [ ]:
fig_legend, ax_legend = plot_kleiner_condition_legend()
plt.show()

In [ ]:
fig_A, ax_A = plot_kleiner_intensity_fractions(
    kleiner_peptide_class_fractions,
    title="Fractions of total intensity",
)
plt.show()

In [ ]:
fig_B, ax_B = plot_kleiner_distinct_counts(
    kleiner_peptide_class_counts,
    title="Counts of distinct peptides",
)
plt.show()

In [ ]:
fig_C, ax_C = plot_kleiner_peptide_sharing(
    kleiner_peptide_sharing_counts,
    title="Peptide sharing",
)
plt.show()

In [ ]:
fig_D, axes_D = plot_kleiner_taxon_counts(
    kleiner_taxon_count_tables["P"],
    kleiner_panel_ref_series["P"],
    condition="P",
    order=KLEINER_ORDER_COMPLETE["P"],
    title="P",
)
plt.show()

In [ ]:
fig_E, axes_E = plot_kleiner_taxon_counts(
    kleiner_taxon_count_tables["U"],
    kleiner_panel_ref_series["U"],
    condition="U",
    order=KLEINER_ORDER_COMPLETE["U"],
    title="U",
)
plt.show()

In [ ]:
fig_F, axes_F = plot_kleiner_taxon_counts(
    kleiner_taxon_count_tables["C"],
    kleiner_panel_ref_series["C"],
    condition="C",
    order=KLEINER_ORDER_COMPLETE["C"],
    title="C",
)
plt.show()

## 9. Load Kleiner method result matrices

In [ ]:
kleiner_results = load_method_condition_tables(
    TAXON_TABLE_DIR,
    KLEINER_METHOD_FILES,
    normalize_columns=False,
)

data_by_method = kleiner_results

## 10. Plot Kleiner method community compositions

In [ ]:
ref_by_cond = {
    "P": 1 / len(KLEINER_ORDER),
    "U": U_exp,
    "C": C_exp,
}
lt2_div_by_cond = {"P": 3.0}
cond_overrides = {
    "P": {"ytick_step": 0.05, "ymax_tick": 0.35},
}

fig, axes = plot_methods_grid(
    data_by_method=data_by_method,
    desired_order=KLEINER_ORDER,
    ref_by_cond=ref_by_cond,
    lt2_div_by_cond=lt2_div_by_cond,
    cond_overrides=cond_overrides,
    panel_size=(3.0, 2.5),
    dpi=400,
    row_gap=0.4,
    col_gap=0.15,
    fs_title=12,
    fs_metrics=8,
    fs_ticks=5.5,
    bar_color="#5799C7",
)
plt.show()

## 11. Compute Kleiner fold-change errors

In [ ]:
kleiner_fc_results = {}
kleiner_fc_summary_rows = []

for method in KLEINER_METHODS:
    kleiner_fc_results[method] = {}
    for contrast, cfg in KLEINER_CONTRASTS.items():
        result = compute_kleiner_logfc_result(
            kleiner_results[method],
            kleiner_ref_props,
            numerator=cfg["numerator"],
            denominator=cfg["denominator"],
            order=KLEINER_ORDER,
        )
        kleiner_fc_results[method][contrast] = result

        err = result["err"]
        kleiner_fc_summary_rows.append({
            "method": method,
            "contrast": contrast,
            "n": int(err.shape[0]),
            "RMSE": float(np.sqrt(np.mean(err.to_numpy() ** 2))),
            "bias": float(np.median(err.to_numpy())),
        })

## 12. Plot Kleiner fold-change error panels

In [ ]:
kleiner_fc_plot_stats = {}

for method in KLEINER_METHODS:
    kleiner_fc_plot_stats[method] = {}
    for contrast, cfg in KLEINER_CONTRASTS.items():
        result = kleiner_fc_results[method][contrast]
        stats = plot_err_vs_P(
            result["numer_obs"],
            result["err"],
            title=f"{method}: {cfg['title']}",
            top_n_labels=0,
            xmin=-4.5,
            xmax=3.0,
            ymin=0,
            ymax=0.2,
            figsize=(2.5, 2.0),
            dpi=400,
            label_taxa=("BXL", "LT2"),
            top_k_by_err=20,
            y_label=cfg["obs_label"],
            fs_title=12,
            fs_axis_labels=9,
            fs_tick_labels=6,
            fs_metrics=8,
            fs_point_labels=6,
            x_label=None,
        )
        kleiner_fc_plot_stats[method][contrast] = stats

# C: Zhao 2023

## 13. Load Zhao method result matrices

In [ ]:
zhao_results = load_method_condition_tables(
    TAXON_TABLE_DIR,
    ZHAO_METHOD_FILES,
    normalize_columns=True,
)

## 14. Compute Zhao all-pair fold changes

In [ ]:
zhao_fc_by_method = {
    method: compute_zhao_foldchanges(sample_tables, ZHAO_CONTRASTS)
    for method, sample_tables in zhao_results.items()
}

zhao_author_fc = {
    contrast: make_zhao_author_fc(
        ZHAO_AUTHOR_OFFSETS[contrast],
        ZHAO_EXPECTED_FC[contrast],
        ZHAO_ORDER,
    )
    for contrast in ZHAO_CONTRASTS
}

zhao_plot_datasets = {
    contrast: {
        "Proteins": zhao_author_fc[contrast],
        "Uniques": zhao_fc_by_method["Uniques"][contrast],
        "Uniform": zhao_fc_by_method["Uniform"][contrast],
        "Clades": zhao_fc_by_method["Clades"][contrast],
        "TaxonLFQ": zhao_fc_by_method["TaxonLFQ"][contrast],
    }
    for contrast in ZHAO_CONTRASTS
}

## 15. Plot Zhao ΔlogFC comparison panels

In [ ]:
zhao_plot_stats = {}

for contrast, cfg in ZHAO_CONTRASTS.items():
    anchor = (0.001, 0.001) if contrast == "S3/S1" else (0, 0)
    fig, ax, stats = plot_delta_logfc_sources_markers(
        datasets=zhao_plot_datasets[contrast],
        expected=ZHAO_EXPECTED_FC[contrast],
        title=cfg["title"],
        desired_order=ZHAO_ORDER,
        y_min=-3.0,
        y_max=3.0,
        log_base=2,
        figsize=(11, 4.5),
        title_fs=13,
        axis_label_fs=11,
        tick_fs=9,
        anno_fs=7.5,
        rmse_box_loc="lower left",
        rmse_box_decimals=3,
        rmse_box_anchor=anchor,
        rmse_box_fs=8,
        rmse_box_title=None,
        rmse_box_edgecolor="0.5",
        rmse_box_facecolor="white",
        rmse_box_alpha=1,
        legend_include_bias_sigma=True,
        legend_as_table=True,
        legend_hide_frame=False,
        show_n_in_legend=False,
    )
    zhao_plot_stats[contrast] = stats